# Replay: Backtest and Live

Stored time-series data can be consumed at two speeds: **max-speed** for
offline backtesting (no sleeping, as fast as the CPU allows) and
**wall-clock replay** where sleeps are injected between events so that
downstream components see realistic inter-arrival gaps.  screamer's `pace`
function handles both modes with a single API: it merges N streams in key
order and emits `(key, value, source)` tuples, optionally sleeping between
events proportional to the key delta divided by `speed`.  Critically, the
**values and their order are identical in both modes** — only the timing
differs.  This is how `backtest then go live` stays honest.

In [ ]:
import numpy as np
from screamer import pace

# Two stored series: keys are logical timestamps (e.g. seconds since epoch)
a = (np.array([0, 10, 30], dtype=np.int64), np.array([1.0, 2.0, 3.0]))
b = (np.array([5, 20],     dtype=np.int64), np.array([0.5, 2.5]))


async def drain(agen):
    """Collect all events from an async generator into a list."""
    out = []
    async for e in agen:
        out.append(e)
    return out

## Turning stored series into an event stream

`pace` is essentially `merge` plus optional pacing.  It takes N
`(keys, values)` pairs, merges them in key order, and yields
`(key, value, source)` tuples — one per event.  With `speed=inf` no
sleeping occurs and events are emitted as fast as the CPU can produce
them.  This is the backtest path.

In [ ]:
# Backtest: max speed, key-sorted, no sleeping
# IPython 7+ supports top-level await; no asyncio.run() needed in a notebook
backtest = await drain(pace(a, b, speed=float("inf")))

print("backtest events (key, value, source):")
for e in backtest:
    print(f"  key={e[0]:3d}  value={e[1]}  source={e[2]}")

## Wall-clock replay with an injectable sleep

With a finite `speed`, `pace` awaits `sleep(key_delta / speed)` between
consecutive events.  In production this is `asyncio.sleep`; for testing and
demos you can inject a fake sleep that records how long it would have waited
without actually blocking.  This makes the demo deterministic and instant
while still demonstrating the timing logic.

In [ ]:
# Wall-clock replay at speed=2.0 with an injectable fake clock
slept = []


async def fake_sleep(sec):
    """Record requested sleep durations without actually sleeping."""
    slept.append(sec)


replay = await drain(pace(a, b, speed=2.0, sleep=fake_sleep))

print("replay events (key, value, source):")
for e in replay:
    print(f"  key={e[0]:3d}  value={e[1]}  source={e[2]}")

print()
print("requested sleeps (key_delta / speed):", slept)

## Values are identical — only timing differs

The core guarantee: `pace` never reorders or modifies events.  The
`(key, value)` pairs are the same regardless of `speed`; only the inter-event
delays change.  A strategy coded against the backtest stream will see
exactly the same sequence of numbers when deployed against the live replay.

In [ ]:
# (key, value) pairs are identical in both modes
assert [e[:2] for e in backtest] == [e[:2] for e in replay], \
    "backtest and replay must produce the same (key, value) pairs"

# sources are also the same (same merge order)
assert [e[2] for e in backtest] == [e[2] for e in replay]

print("events (key, value):      ", [(e[0], e[1]) for e in backtest])
print("sleeps in replay mode:    ", slept)

# Verify sleep values: key_delta / speed=2.0
merged_keys = [e[0] for e in backtest]
deltas = [merged_keys[i+1] - merged_keys[i] for i in range(len(merged_keys)-1)]
expected_sleeps = [d / 2.0 for d in deltas]
assert slept == expected_sleeps, "sleep = key_delta / speed"

print("key-deltas in merged stream:", deltas)
print("expected sleeps (delta/2): ", expected_sleeps)
print("backtest == replay: PASSED")

**Takeaways**

- `pace` = `merge` + optional pacing: it yields key-sorted
  `(key, value, source)` events from N input streams.
- `speed=inf` gives a pure backtest with no sleeping; a finite `speed`
  injects proportional sleeps so wall-clock time tracks key-delta time.
- The `sleep` parameter is injectable — replace `asyncio.sleep` with a fake
  to make replay deterministic and instant in tests.
- Keys must be **metric** (subtractable) for wall-clock pacing: nanoseconds,
  seconds, bar indices, etc.
- Values and order are **identical** in backtest and replay: a strategy that
  passes backtesting will see the same number sequence in production.
- In a Jupyter notebook use top-level `await` (IPython 7+); `asyncio.run()`
  is for scripts where no event loop is running yet.